# Marketing Risk & Resilience Simulation Lab

A coursework notebook for exploring advertising risk with Python.

1. Load and explore advertising data.
2. Simulate campaign ROI with Monte Carlo sampling.
3. Flag unusual observations with Isolation Forest.
4. Score synthetic social posts using local rules or OpenRouter.
5. Export the analysis tables to Excel.

See the data-loading section for the optional advertising dataset. The notebook generates synthetic data when the CSV is unavailable.

---

##  0. Environment Setup

In [ ]:
# Run this cell first to install the notebook dependencies.
%pip install pandas numpy scikit-learn plotly openpyxl scipy requests

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings, os, json
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ All libraries loaded successfully')

---
##  Module 1 — Data Loading & Exploratory Risk Analysis

### Real-World Dataset: Digital Advertising Performance

**Source:** Kaggle — *Evaluating the Effect of Online Ads on Revenue*  
**Download:** `kaggle datasets download harshalhonde/evaluating-the-effect-of-online-ads-on-revenue`  
Or place `advertising.csv` in the same folder. If not found, a synthetic replica is auto-generated.

**Columns we work with:**
- `TV`, `Radio`, `Newspaper` — ad spend by channel ($000s)
- `Sales` — resulting sales ($000s)
- We engineer: `Total_Spend`, `ROI`, `CPR` (cost per revenue unit), `Channel_Mix`

In [ ]:
import os
import numpy as np # Import numpy
import pandas as pd # Import pandas
def load_or_generate_dataset():
    """
    Tries to load real Kaggle advertising dataset.
    Falls back to a realistic synthetic version if not found.
    """
    csv_paths = ['advertising.csv', 'data/advertising.csv', 'Advertising.csv']

    for path in csv_paths:
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f'✅ Loaded real dataset from {path} — {len(df)} rows')
            return df

    # ── Synthetic Replica ───────────────────────────────────────────
    print('⚠️  advertising.csv not found — generating synthetic replica...')
    print('   (Download from: https://www.kaggle.com/datasets/harshalhonde/evaluating-the-effect-of-online-ads-on-revenue)')

    np.random.seed(42)
    n = 400
    dates = pd.date_range('2021-01-01', periods=n, freq='W')

    TV        = np.random.lognormal(np.log(150), 0.6, n).clip(5, 300)
    Radio     = np.random.lognormal(np.log(25), 0.5, n).clip(1, 50)
    Newspaper = np.random.lognormal(np.log(20), 0.8, n).clip(1, 80)

    # Sales = realistic non-linear response + noise
    Sales = (0.047 * TV + 0.188 * Radio + 0.003 * Newspaper
             + 0.0005 * TV * Radio  # interaction effect
             + np.random.normal(0, 1.2, n)).clip(1)

    df = pd.DataFrame({
        'Date': dates, 'TV': TV, 'Radio': Radio,
        'Newspaper': Newspaper, 'Sales': Sales
    })
    print(f'✅ Synthetic dataset generated — {len(df)} rows')
    return df

df_raw = load_or_generate_dataset()
df_raw.head()

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────
df = df_raw.copy()

# Total spend and ROI
df['Total_Spend']   = df['TV'] + df['Radio'] + df['Newspaper']
df['ROI']           = ((df['Sales'] * 1000 - df['Total_Spend'] * 1000)
                        / (df['Total_Spend'] * 1000) * 100)  # %
df['CPR']           = df['Total_Spend'] / df['Sales']  # cost per revenue unit
df['TV_Share']      = df['TV'] / df['Total_Spend'] * 100
df['Radio_Share']   = df['Radio'] / df['Total_Spend'] * 100

# Rolling efficiency (4-week window)
df['ROI_4w_avg']    = df['ROI'].rolling(4, min_periods=1).mean()
df['Spend_4w_avg']  = df['Total_Spend'].rolling(4, min_periods=1).mean()

# Risk flag: ROI below 10th percentile
roi_p10 = df['ROI'].quantile(0.10)
df['Low_ROI_Flag']  = (df['ROI'] < roi_p10).astype(int)

print('📊 Engineered features:')
print(df[['Total_Spend', 'ROI', 'CPR', 'TV_Share', 'Low_ROI_Flag']].describe().round(2))

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── EDA: Channel Spend vs Sales (Interactive) ────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['TV Spend vs Sales', 'Radio Spend vs Sales',
                    'ROI Distribution', 'Channel Mix Over Time']
)

fig.add_trace(go.Scatter(x=df['TV'], y=df['Sales'], mode='markers',
    marker=dict(color=df['ROI'], colorscale='RdYlGn', size=6, opacity=0.7,
                showscale=True, colorbar=dict(title='ROI%', x=0.45)),
    name='TV', hovertemplate='TV: $%{x}k<br>Sales: $%{y}k<br>ROI: %{marker.color:.1f}%'),
    row=1, col=1)

fig.add_trace(go.Scatter(x=df['Radio'], y=df['Sales'], mode='markers',
    marker=dict(color='#7c6af7', size=6, opacity=0.7),
    name='Radio'), row=1, col=2)

fig.add_trace(go.Histogram(x=df['ROI'], nbinsx=40, name='ROI',
    marker_color='#00e5c0', opacity=0.8), row=2, col=1)
fig.add_vline(x=df['ROI'].mean(), line_dash='dash', line_color='white',
              annotation_text=f'Mean {df["ROI"].mean():.1f}%', row=2, col=1)

# Channel mix area chart
fig.add_trace(go.Scatter(x=df.index, y=df['TV_Share'], fill='tozeroy',
    name='TV %', fillcolor='rgba(255,79,94,0.3)', line_color='#ff4f5e'), row=2, col=2)
fig.add_trace(go.Scatter(x=df.index, y=df['Radio_Share'], fill='tozeroy',
    name='Radio %', fillcolor='rgba(124,106,247,0.3)', line_color='#7c6af7'), row=2, col=2)

fig.update_layout(height=700, template='plotly_dark',
    title_text='📊 Marketing Performance EDA', showlegend=True)
fig.show()
print(f'\n🚨 Low-ROI weeks flagged: {df["Low_ROI_Flag"].sum()} / {len(df)}')

---
##  Module 2 — Monte Carlo Campaign Risk Simulation

We fit empirical distributions to the **real dataset** parameters, then simulate 10,000 future campaign scenarios to estimate ROI risk.

**Key metrics:**
- **VaR(5%)** — worst-case ROI in 5% of scenarios
- **CVaR** — average ROI in the worst 5% (expected shortfall)
- **Probability of Loss** — P(ROI < 0%)

**Task:** Complete the `compute_risk_metrics()` and `plot_tornado()` functions.

In [ ]:
from scipy import stats

# ── Fit distributions to real data ───────────────────────────────
# Fit log-normal to total spend
spend_ln_params = stats.lognorm.fit(df['Total_Spend'], floc=0)
# Fit beta to ROI (normalized to [0,1])
roi_min, roi_max = df['ROI'].min() - 1, df['ROI'].max() + 1
roi_norm = (df['ROI'] - roi_min) / (roi_max - roi_min)
beta_params = stats.beta.fit(roi_norm.clip(0.001, 0.999), floc=0, fscale=1)

print('📐 Fitted distributions (from real data):')
print(f'  Total Spend  → LogNormal(μ={spend_ln_params[2]:.1f}, σ={spend_ln_params[0]:.2f})')
print(f'  ROI (norm)   → Beta(α={beta_params[0]:.2f}, β={beta_params[1]:.2f})')

In [ ]:
N_SIM = 10_000

def simulate_campaigns(n=N_SIM):
    """Monte Carlo simulation using empirically-fitted distributions."""
    spend   = stats.lognorm(*spend_ln_params).rvs(n)
    # ROI driven by spend + channel mix + external market noise
    tv_share = np.random.beta(3, 1.5, n) * 0.75  # TV usually 40-75%
    radio_eff = np.random.beta(2, 5, n)           # Radio efficiency
    mkt_noise = np.random.normal(1.0, 0.15, n)    # market conditions

    # Simulate revenue using fitted regression coefficients
    tv_spend    = spend * tv_share
    radio_spend = spend * (1 - tv_share) * 0.6
    revenue     = (0.047 * tv_spend + 0.188 * radio_spend) * 1000 * mkt_noise
    roi_sim     = (revenue - spend * 1000) / (spend * 1000) * 100
    return roi_sim, spend

roi_sim, spend_sim = simulate_campaigns()
print(f'✅ Simulated {N_SIM:,} campaigns')
print(f'   Mean ROI:  {roi_sim.mean():.1f}%')
print(f'   Std ROI:   {roi_sim.std():.1f}%')

In [ ]:
def compute_risk_metrics(roi_array):
    """
    ✏️  STUDENT TASK: Compute VaR, CVaR, and loss probability.

    Args:
        roi_array: np.array of simulated ROI values (%)
    Returns:
        dict with keys: var_5, var_10, cvar, prob_loss, best_case, worst_case
    """
    # ── TODO: Fill in each metric ──────────────────────────────────
    var_5      = np.percentile(roi_array, 5)        # 5th percentile
    var_10     = np.percentile(roi_array, 10)
    cvar       = roi_array[roi_array <= var_5].mean()  # Expected Shortfall
    prob_loss  = (roi_array < 0).mean() * 100
    best_case  = np.percentile(roi_array, 95)
    worst_case = roi_array.min()
    # ────────────────────────────────────────────────────────────────
    return {
        'var_5': var_5, 'var_10': var_10, 'cvar': cvar,
        'prob_loss': prob_loss, 'best_case': best_case, 'worst_case': worst_case
    }

metrics = compute_risk_metrics(roi_sim)
print('\n📊 Risk Metrics (from Monte Carlo):')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.2f}%' if 'prob' not in k else f'  {k:12s}: {v:.2f}% of scenarios')

In [ ]:
# ── Risk Distribution Plot ────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=roi_sim, nbinsx=80, name='Simulated ROI',
    marker=dict(color='rgba(0,229,192,0.6)', line=dict(width=0)),
    histnorm='probability density'
))

# Overlay KDE
kde_x = np.linspace(roi_sim.min(), roi_sim.max(), 300)
kde   = stats.gaussian_kde(roi_sim)
fig.add_trace(go.Scatter(x=kde_x, y=kde(kde_x), name='KDE',
    line=dict(color='#00e5c0', width=2)))

# Vertical markers
for label, val, color in [
    (f'VaR(5%) = {metrics["var_5"]:.1f}%',  metrics['var_5'],  '#ff4f5e'),
    (f'CVaR = {metrics["cvar"]:.1f}%',       metrics['cvar'],   '#ffb347'),
    (f'Mean = {roi_sim.mean():.1f}%',        roi_sim.mean(),    '#7c6af7'),
]:
    fig.add_vline(x=val, line_dash='dash', line_color=color,
                  annotation_text=label, annotation_position='top')

# Shade loss region
loss_mask = kde_x < 0
if loss_mask.any():
    fig.add_trace(go.Scatter(
        x=np.concatenate([[kde_x[loss_mask][0]], kde_x[loss_mask], [kde_x[loss_mask][-1]]]),
        y=np.concatenate([[0], kde(kde_x[loss_mask]), [0]]),
        fill='toself', fillcolor='rgba(255,79,94,0.15)',
        line=dict(width=0), name='Loss Zone'
    ))

fig.update_layout(
    title=f'Monte Carlo ROI Distribution (n={N_SIM:,}) | P(Loss) = {metrics["prob_loss"]:.1f}%',
    xaxis_title='ROI (%)', yaxis_title='Probability Density',
    template='plotly_dark', height=450, showlegend=True
)
fig.show()

In [ ]:
# ── Tornado Sensitivity Chart ─────────────────────────────────────
# ✏️  STUDENT TASK: Vary each parameter ±20% and measure impact on mean ROI

base_roi = roi_sim.mean()
params = {
    'Market Conditions (±20%)':    (0.8, 1.2),
    'TV Spend Allocation (±20%)':  (0.8, 1.2),
    'Radio Efficiency (±20%)':     (0.8, 1.2),
    'Total Budget (±20%)':         (0.8, 1.2),
    'Newspaper Efficiency (±20%)': (0.8, 1.2),
}

# Simplified sensitivity (students should replace with proper parametric runs)
sensitivities = []
impact_factors = [18.4, 12.1, 8.7, 6.2, 1.8]  # % ROI change per parameter
for (param, _), impact in zip(params.items(), impact_factors):
    sensitivities.append({'Parameter': param,
                          'Low': base_roi - impact,
                          'High': base_roi + impact,
                          'Range': impact * 2})

sens_df = pd.DataFrame(sensitivities).sort_values('Range')

fig_tornado = go.Figure()
fig_tornado.add_trace(go.Bar(
    y=sens_df['Parameter'], x=sens_df['Range'],
    orientation='h',
    marker=dict(
        color=sens_df['Range'],
        colorscale=[[0, '#7c6af7'], [0.5, '#ffb347'], [1, '#ff4f5e']]
    ),
    text=[f'±{v:.1f}%' for v in sens_df['Range']/2],
    textposition='inside'
))
fig_tornado.update_layout(
    title='🌪️ Tornado Chart — ROI Sensitivity to Key Risk Factors',
    xaxis_title='ROI Impact Range (%)', template='plotly_dark', height=380
)
fig_tornado.show()
print('\n✏️  Extension: Re-run each parameter using actual Monte Carlo runs (vary the distribution params above)')

---
##  Module 3 — ML Anomaly Detection on Real Campaign Data

We apply **Isolation Forest** to the advertising dataset time-series to automatically detect weeks where marketing performance deviates abnormally — potential signals of competitor activity, budget misallocation, or channel saturation.

**Task:** Engineer rolling features, train the model, detect anomalies, and explain what drove each flag.

In [ ]:
def build_risk_features(df):
    """
    ✏️  STUDENT TASK: Engineer features that capture spend/performance anomalies.
    """
    f = df.copy()

    # Rolling z-scores (deviation from 4-week baseline)
    for col in ['Total_Spend', 'Sales', 'ROI', 'CPR']:
        roll_mean = f[col].rolling(4, min_periods=2).mean()
        roll_std  = f[col].rolling(4, min_periods=2).std().replace(0, 1e-6)
        f[f'{col}_zscore'] = (f[col] - roll_mean) / roll_std

    # Rate of change features
    f['Spend_pct_change']  = f['Total_Spend'].pct_change().fillna(0)
    f['Sales_pct_change']  = f['Sales'].pct_change().fillna(0)
    f['ROI_pct_change']    = f['ROI'].pct_change().fillna(0)

    # Channel imbalance (sudden TV/Radio ratio shift)
    f['TV_Radio_Ratio']    = (f['TV'] / (f['Radio'] + 0.01)).pct_change().fillna(0)

    # Spend efficiency drop
    f['Efficiency_Drop']   = f['Sales'] / f['Total_Spend']

    feature_cols = [
        'Total_Spend_zscore', 'Sales_zscore', 'ROI_zscore', 'CPR_zscore',
        'Spend_pct_change', 'Sales_pct_change', 'ROI_pct_change',
        'TV_Radio_Ratio', 'Efficiency_Drop'
    ]
    return f.dropna(), feature_cols

df_feat, feature_cols = build_risk_features(df)
print(f'✅ Feature matrix: {df_feat.shape[0]} rows × {len(feature_cols)} features')
print(f'   Features: {", ".join(feature_cols)}')

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# ── Train Isolation Forest ────────────────────────────────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_feat[feature_cols])

iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # expect ~5% anomalous weeks
    random_state=42,
    max_features=0.8
)
iso.fit(X_scaled)

df_feat['anomaly_score']  = iso.decision_function(X_scaled)  # lower = more anomalous
df_feat['anomaly_label']  = iso.predict(X_scaled)             # -1 = anomaly, 1 = normal
df_feat['is_anomaly']     = df_feat['anomaly_label'] == -1

anomaly_count = df_feat['is_anomaly'].sum()
print(f'✅ Model trained | Anomalies detected: {anomaly_count} weeks ({anomaly_count/len(df_feat)*100:.1f}%)')
print('\n🚨 Top 10 most anomalous weeks:')
top_anomalies = df_feat.nsmallest(10, 'anomaly_score')[['Total_Spend', 'Sales', 'ROI', 'anomaly_score']]
print(top_anomalies.to_string())

In [ ]:
# ── Anomaly Timeline Visualization ───────────────────────────────
normal    = df_feat[~df_feat['is_anomaly']]
anomalies = df_feat[df_feat['is_anomaly']]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=['Sales Performance', 'Total Ad Spend', 'Anomaly Score'],
    vertical_spacing=0.08)

# Sales
fig.add_trace(go.Scatter(x=normal.index, y=normal['Sales'],
    mode='lines', name='Sales (Normal)', line=dict(color='#00e5c0', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=anomalies.index, y=anomalies['Sales'],
    mode='markers', name='🚨 Anomaly',
    marker=dict(color='#ff4f5e', size=10, symbol='x', line=dict(width=2))), row=1, col=1)

# Spend
fig.add_trace(go.Scatter(x=df_feat.index, y=df_feat['Total_Spend'],
    mode='lines', name='Spend', line=dict(color='#7c6af7', width=1.5),
    fill='tozeroy', fillcolor='rgba(124,106,247,0.1)'), row=2, col=1)
fig.add_trace(go.Scatter(x=anomalies.index, y=anomalies['Total_Spend'],
    mode='markers', name='Anomaly Spend',
    marker=dict(color='#ff4f5e', size=10, symbol='x'), showlegend=False), row=2, col=1)

# Anomaly score bar
fig.add_trace(go.Bar(x=df_feat.index, y=-df_feat['anomaly_score'],
    name='Anomaly Score', marker=dict(
        color=np.where(df_feat['is_anomaly'], '#ff4f5e', '#2a2a3d'))), row=3, col=1)

fig.update_layout(height=650, template='plotly_dark',
    title='🤖 Isolation Forest — Anomaly Detection on Campaign Data', showlegend=True)
fig.show()

In [ ]:
# ── Anomaly Explanation ───────────────────────────────────────────
print('🔍 Anomaly Root Cause Analysis')
print('=' * 60)

baseline = df_feat[~df_feat['is_anomaly']][feature_cols].mean()

for idx, row in df_feat[df_feat['is_anomaly']].head(5).iterrows():
    deviations = ((row[feature_cols] - baseline) / (baseline.abs() + 1e-6) * 100)
    top_driver = deviations.abs().idxmax()
    direction  = '📈 spike' if deviations[top_driver] > 0 else '📉 drop'
    print(f'\nWeek {idx:>3} | ROI: {row["ROI"]:>7.1f}% | Spend: ${row["Total_Spend"]:>6.1f}k | '
          f'Score: {row["anomaly_score"]:>6.3f}')
    print(f'  → Primary driver: {top_driver} ({direction}, {abs(deviations[top_driver]):.0f}% from baseline)')

print('\n✏️  Extension Task: Compare IsolationForest vs LocalOutlierFactor results')

---
## Module 4 — Reputation Risk Scoring

The notebook scores synthetic social posts using a local keyword-based fallback by default. The optional OpenRouter route reads `OPENROUTER_API_KEY` from the environment of the Jupyter process. Set it before launching Jupyter to try that route.

The scores feed a weighted Brand Risk Index and a table of the highest-risk posts.

In [ ]:
# ── Generate Synthetic Social Posts ──────────────────────────────
# Simulates a social listening feed for brand 'NovaByte'
posts_data = [
    # Normal positive posts
    {'id': 1,  'text': 'NovaByte ads are everywhere this week. Pretty effective campaign honestly.', 'type': 'normal'},
    {'id': 2,  'text': 'Just bought from NovaByte after seeing their ad. Really smooth experience!', 'type': 'normal'},
    {'id': 3,  'text': 'NovaByte radio ad got stuck in my head. Good jingle!', 'type': 'normal'},
    {'id': 4,  'text': 'Saw NovaByte on TV last night. The new campaign looks slick.', 'type': 'normal'},
    {'id': 5,  'text': 'NovaByte discount code worked, 10% off my order. Happy customer.', 'type': 'normal'},
    {'id': 6,  'text': 'NovaByte keeps popping up in my feed. Decent targeting.', 'type': 'normal'},
    {'id': 7,  'text': 'Their customer service team actually responded fast. Impressed NovaByte.', 'type': 'normal'},
    {'id': 8,  'text': 'Quarterly results show NovaByte marketing ROI up 12% YoY.', 'type': 'normal'},
    {'id': 9,  'text': 'Been using NovaByte for 2 years. Quality never drops.', 'type': 'normal'},
    {'id': 10, 'text': 'The new NovaByte TV ad is really well produced. Bold choice.', 'type': 'normal'},
    # Moderate risk
    {'id': 11, 'text': 'Why is NovaByte spending so much on ads but their product quality is slipping?', 'type': 'moderate'},
    {'id': 12, 'text': 'NovaByte pricing went up again. Not sure the value is there anymore.', 'type': 'moderate'},
    {'id': 13, 'text': 'Seeing a lot of NovaByte ads but delivery times have gotten worse.', 'type': 'moderate'},
    {'id': 14, 'text': 'NovaByte customer support is overwhelmed. 3-day wait for a response.', 'type': 'moderate'},
    {'id': 15, 'text': 'Multiple friends say NovaByte products broke within 6 months. Concerning trend.', 'type': 'moderate'},
    # High risk / crisis signals
    {'id': 16, 'text': 'BREAKING: Multiple users reporting NovaByte data breach — change your passwords NOW', 'type': 'crisis'},
    {'id': 17, 'text': 'NovaByte just quietly recalled batch #A4-2024. No public announcement. This is shady.', 'type': 'crisis'},
    {'id': 18, 'text': 'Major influencer @TechReview just posted a devastating NovaByte review. 2M views already.', 'type': 'crisis'},
    {'id': 19, 'text': 'Lawsuit filed against NovaByte for misleading advertising claims. Court docs leaked.', 'type': 'crisis'},
    {'id': 20, 'text': '#NovaByteLied is trending. Customers saying product specs were fabricated in ads.', 'type': 'crisis'},
    {'id': 21, 'text': 'Ex-NovaByte employee says the company faked environmental certifications. Thread below 🧵', 'type': 'crisis'},
    {'id': 22, 'text': 'NovaByte stock down 8% after reports of false advertising investigation by FTC.', 'type': 'crisis'},
    # Recovery signals
    {'id': 23, 'text': 'NovaByte CEO just issued a public apology. Acknowledges the product issues.', 'type': 'recovery'},
    {'id': 24, 'text': 'NovaByte offering full refunds now. At least they are taking action.', 'type': 'recovery'},
    {'id': 25, 'text': 'NovaByte hired a new Chief Product Officer. Hopefully a sign of change.', 'type': 'recovery'},
]

posts_df = pd.DataFrame(posts_data)
print(f'✅ Social listening dataset: {len(posts_df)} posts')
print(posts_df['type'].value_counts().to_string())

In [ ]:
# ── LLM Risk Scorer ───────────────────────────────────────────────
# ✏️  STUDENT TASK: Complete the API call and JSON parsing
import os
import json
import requests

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()

USE_LLM = bool(OPENROUTER_API_KEY)
if USE_LLM:
    print("✅ OpenRouter API key detected")
else:
    print("⚠️ OpenRouter API key not found — using rule-based fallback")

SYSTEM_PROMPT = """You are a senior marketing risk analyst specializing in brand reputation.
For each social media post provided, respond ONLY with a valid JSON object (no prose, no markdown):
{
  "sentiment": "positive" | "neutral" | "negative",
  "risk_level": <integer 0-10>,
  "risk_category": "none" | "product" | "reputation" | "legal" | "viral" | "data_breach",
  "urgency": "low" | "medium" | "high" | "critical",
  "reasoning": "<one sentence>"
}
Risk level guide: 0=no risk, 3=minor concern, 6=significant risk, 9-10=brand crisis."""

def analyze_post_llm(text):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openrouter/auto",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f'Analyze this post: "{text}"'}
        ],
        "temperature": 0,
        "response_format": {"type": "json_object"}
    }

    response = requests.post(url, headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    content = data["choices"][0]["message"]["content"]
    return json.loads(content)

def analyze_post_fallback(text):
    crisis_kws = ['breach', 'lawsuit', 'recall', 'trending', 'leaked', 'faked', 'lying', 'lied', 'ftc']
    moderate_kws = ['slipping', 'worse', 'broke', 'overwhelmed', 'wait']
    positive_kws = ['great', 'smooth', 'happy', 'impressed', 'slick', 'discount', 'quick']
    text_l = text.lower()
    if any(k in text_l for k in crisis_kws):
        return {'sentiment': 'negative', 'risk_level': 8, 'risk_category': 'reputation',
                'urgency': 'critical', 'reasoning': 'Crisis keyword detected'}
    elif any(k in text_l for k in moderate_kws):
        return {'sentiment': 'negative', 'risk_level': 4, 'risk_category': 'product',
                'urgency': 'medium', 'reasoning': 'Quality concern detected'}
    elif any(k in text_l for k in positive_kws):
        return {'sentiment': 'positive', 'risk_level': 0, 'risk_category': 'none',
                'urgency': 'low', 'reasoning': 'Positive sentiment'}
    else:
        return {'sentiment': 'neutral', 'risk_level': 1, 'risk_category': 'none',
                'urgency': 'low', 'reasoning': 'Neutral content'}

print("Functions defined. Run next cell to score all posts.")

In [ ]:
# ── Score All Posts ───────────────────────────────────────────────
results = []
analyze_fn = analyze_post_llm if USE_LLM else analyze_post_fallback

for _, row in posts_df.iterrows():
    try:
        result = analyze_fn(row['text'])
        result['post_id']   = row['id']
        result['text']      = row['text']
        result['true_type'] = row['type']
        results.append(result)
    except Exception as e:
        print(f'  Error on post {row["id"]}: {e}')

results_df = pd.DataFrame(results)
print(f'✅ Scored {len(results_df)} posts')
print('\nRisk level by true post type:')
print(results_df.groupby('true_type')['risk_level'].agg(['mean','max']).round(1).to_string())

In [ ]:
# ── Brand Risk Index (BRI) Computation ───────────────────────────
def compute_bri(results_df):
    """
    ✏️  STUDENT TASK: Build a weighted Brand Risk Index (0-100).
    Weights: critical urgency = 3x, high = 2x, medium = 1.5x, low = 1x
    Penalty: viral + legal categories get +2 bonus risk points
    """
    urgency_weights = {'low': 1.0, 'medium': 1.5, 'high': 2.0, 'critical': 3.0}
    category_penalty = {'viral': 2, 'legal': 2, 'data_breach': 3, 'reputation': 1}

    df = results_df.copy()
    df['weight']    = df['urgency'].map(urgency_weights).fillna(1)
    df['penalty']   = df['risk_category'].map(category_penalty).fillna(0)
    df['adj_risk']  = (df['risk_level'] + df['penalty']).clip(0, 10)
    df['w_risk']    = df['adj_risk'] * df['weight']

    bri = (df['w_risk'].sum() / (df['weight'].sum() * 10)) * 100

    top_risk = df.nlargest(3, 'adj_risk')[['text','risk_level','risk_category','urgency']]

    status = '🟢 LOW' if bri < 30 else ('🟡 MODERATE' if bri < 60 else '🔴 HIGH RISK')
    print(f'\n📊 Brand Risk Index (BRI): {bri:.1f}/100  {status}')
    print(f'   Posts analysed: {len(df)} | Crisis posts: {(df["risk_level"] >= 7).sum()}')
    print(f'\n🚨 Top 3 Risk Posts:')
    for _, r in top_risk.iterrows():
        print(f'  [{r["risk_level"]}/10] [{r["urgency"].upper()}] {r["text"][:80]}...')

    return bri, df

bri_score, scored_df = compute_bri(results_df)

In [ ]:
# ── BRI Visualization ─────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Risk Level by Category', 'Sentiment Distribution'],
    specs=[[{'type': 'bar'}, {'type': 'pie'}]])

cat_risk = scored_df.groupby('risk_category')['risk_level'].mean().sort_values(ascending=False)
colors   = ['#ff4f5e' if v > 6 else '#ffb347' if v > 3 else '#00e5c0' for v in cat_risk.values]

fig.add_trace(go.Bar(x=cat_risk.index, y=cat_risk.values,
    marker_color=colors, name='Avg Risk Level'), row=1, col=1)

sent_counts = scored_df['sentiment'].value_counts()
fig.add_trace(go.Pie(
    labels=sent_counts.index, values=sent_counts.values,
    hole=0.4,
    marker=dict(colors=['#00e5c0', '#7c6af7', '#ff4f5e'])), row=1, col=2)

fig.update_layout(height=400, template='plotly_dark',
    title=f'Brand Risk Dashboard | BRI: {bri_score:.1f}/100')
fig.show()

---
##  Module 5 — Power BI Export & Dashboard Task

This module exports all analysis results to structured Excel files optimized for Power BI import.

### What you'll build in Power BI:
1. **Campaign Performance Overview** — KPI cards, spend vs. sales trend line
2. **Risk Heatmap** — ROI volatility by channel and time period  
3. **Anomaly Timeline** — Highlighted anomaly weeks with drill-through
4. **Brand Risk Gauge** — BRI score with traffic light conditional formatting
5. **Monte Carlo Summary** — VaR/CVaR summary table with risk band slicers

**Step 1:** Run the cell below to export all sheets  
**Step 2:** Open Power BI Desktop → Get Data → Excel  
**Step 3:** Follow the Power BI task guide at the bottom of this notebook

In [ ]:
# ── Export All Data to Power BI-Ready Excel ───────────────────────
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

EXPORT_PATH = 'marketing_risk_powerbi.xlsx'

# ── Sheet 1: Campaign Performance ────────────────────────────────
df_export = df[['Total_Spend','TV','Radio','Newspaper','Sales','ROI','CPR',
                'TV_Share','Radio_Share','Low_ROI_Flag','ROI_4w_avg']].copy()
df_export.columns = ['Total_Spend','TV_Spend','Radio_Spend','Newspaper_Spend',
                     'Sales','ROI_Pct','CPR','TV_Share_Pct','Radio_Share_Pct',
                     'Low_ROI_Flag','ROI_4W_Rolling']
if 'Date' in df.columns:
    df_export.insert(0, 'Date', df['Date'].values)

# ── Sheet 2: Anomaly Results ──────────────────────────────────────
df_anomaly_export = df_feat[['Total_Spend','Sales','ROI','anomaly_score',
                              'is_anomaly','Spend_pct_change','Sales_pct_change']].copy()
df_anomaly_export['Risk_Level'] = np.where(df_anomaly_export['is_anomaly'], 'HIGH', 'NORMAL')

# ── Sheet 3: Monte Carlo Summary ─────────────────────────────────
mc_summary = pd.DataFrame({
    'Metric': ['Mean ROI', 'Std Dev', 'VaR 5%', 'VaR 10%', 'CVaR', 'P(Loss)', 'Best Case (95th)', 'Worst Case'],
    'Value_Pct': [roi_sim.mean(), roi_sim.std(),
                  metrics['var_5'], metrics['var_10'], metrics['cvar'],
                  metrics['prob_loss'], metrics['best_case'], metrics['worst_case']],
    'Category': ['Performance','Risk','Risk','Risk','Risk','Risk','Opportunity','Risk']
})

# ── Sheet 4: Brand Risk Scores ────────────────────────────────────
brand_export = scored_df[['post_id','text','sentiment','risk_level',
                           'risk_category','urgency','reasoning']].copy()
brand_export['BRI_Score'] = bri_score

# ── Sheet 5: KPI Summary (for Power BI cards) ─────────────────────
kpi_summary = pd.DataFrame({
    'KPI': ['Total Campaigns Analysed', 'Avg ROI', 'VaR 5%', 'Anomalies Detected',
            'Brand Risk Index', 'Low ROI Weeks', 'Best Channel ROI'],
    'Value': [len(df), df['ROI'].mean(), metrics['var_5'],
              int(df_feat['is_anomaly'].sum()), bri_score,
              int(df['Low_ROI_Flag'].sum()),
              df.groupby(lambda x: 'TV' if df['TV_Share'].iloc[x%len(df)] > 60 else 'Radio')['ROI'].mean().max()],
    'Unit': ['weeks', '%', '%', 'weeks', '/100', 'weeks', '%'],
    'Status': ['INFO',
               'GOOD' if df['ROI'].mean() > 20 else 'WARN',
               'RISK' if metrics['var_5'] < -10 else 'WARN',
               'RISK' if df_feat['is_anomaly'].sum() > 10 else 'WARN',
               'RISK' if bri_score > 50 else 'WARN' if bri_score > 25 else 'GOOD',
               'WARN', 'GOOD']
})

# ── Write Excel with formatting ───────────────────────────────────
with pd.ExcelWriter(EXPORT_PATH, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Campaign_Performance', index=False)
    df_anomaly_export.to_excel(writer, sheet_name='Anomaly_Results', index=False)
    mc_summary.to_excel(writer, sheet_name='MonteCarlo_Summary', index=False)
    brand_export.to_excel(writer, sheet_name='Brand_Risk_Scores', index=False)
    kpi_summary.to_excel(writer, sheet_name='KPI_Summary', index=False)

# ── Apply styling ──────────────────────────────────────────────────
wb = openpyxl.load_workbook(EXPORT_PATH)
header_fill   = PatternFill(start_color='1A1A26', end_color='1A1A26', fill_type='solid')
header_font   = Font(color='00E5C0', bold=True, name='Calibri', size=11)
risk_fill     = PatternFill(start_color='FF4F5E', end_color='FF4F5E', fill_type='solid')
warn_fill     = PatternFill(start_color='FFB347', end_color='FFB347', fill_type='solid')
good_fill     = PatternFill(start_color='00E5C0', end_color='00E5C0', fill_type='solid')

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')
    ws.freeze_panes = 'A2'
    for col in ws.columns:
        max_len = max(len(str(c.value or '')) for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

# Colour-code status column in KPI sheet
ws_kpi = wb['KPI_Summary']
for row in ws_kpi.iter_rows(min_row=2, max_col=4):
    status_cell = row[3]
    if status_cell.value == 'RISK':
        status_cell.fill = risk_fill
        status_cell.font = Font(bold=True, color='FFFFFF')
    elif status_cell.value == 'WARN':
        status_cell.fill = warn_fill
        status_cell.font = Font(bold=True)
    elif status_cell.value == 'GOOD':
        status_cell.fill = good_fill
        status_cell.font = Font(bold=True)

wb.save(EXPORT_PATH)
print(f'✅ Exported to: {EXPORT_PATH}')
print('   Sheets: Campaign_Performance | Anomaly_Results | MonteCarlo_Summary | Brand_Risk_Scores | KPI_Summary')
print(f'\n📁 File size: {os.path.getsize(EXPORT_PATH)/1024:.1f} KB')

---
##  Power BI Task Guide

### Setup (5 min)
1. Open **Power BI Desktop** (free download from Microsoft)
2. Click **Home → Get Data → Excel Workbook**
3. Select `marketing_risk_powerbi.xlsx` → load all 5 sheets
4. In Power Query Editor: confirm data types (dates as Date, numerics as Decimal)

---

### Task A — Campaign Performance Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Line Chart | X=Date, Y=Sales, Y2=Total_Spend | Dual axis, 4-week rolling average |
| KPI Card | Value=Avg ROI_Pct | Conditional format: Green>20%, Red<0% |
| Stacked Bar | X=Date, Values=TV_Spend, Radio_Spend, Newspaper_Spend | Channel breakdown |
| Scatter Plot | X=Total_Spend, Y=Sales, Size=ROI_Pct | Bubble chart with Low_ROI_Flag colour |

### Task B — Risk Dashboard Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Gauge | Value=BRI_Score | Min=0, Max=100, Target=30 |
| Table | All columns from KPI_Summary | Conditional format Status column |
| Clustered Bar | X=Metric, Y=Value_Pct from MonteCarlo | Highlight VaR/CVaR bars red |
| Card (3x) | VaR 5%, CVaR, P(Loss) | Font colour = red |

### Task C — Anomaly Monitor Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Line + Scatter | X=Index, Y=Sales, Scatter=is_anomaly | Mark anomaly points red ✕ |
| Bar Chart | X=Index, Y=anomaly_score | Colour by Risk_Level column |
| Slicer | Risk_Level | Dropdown filter: HIGH / NORMAL |
| Drill-through | Click anomaly row → shows post detail card | Set up drill-through filter |

### Task D — Brand Reputation Page *(20 min)*
| Visual | Fields | Configuration |
|--------|--------|---------------|
| Donut Chart | Legend=sentiment, Values=count | Colours: green/grey/red |
| Treemap | Category=risk_category, Size=risk_level | Colour saturation = urgency |
| Table | text, risk_level, urgency, reasoning | Sort by risk_level desc, top 10 |
| Gauge | BRI_Score | Zone bands: 0-30 green, 30-60 amber, 60-100 red |

### Task E — Extension: DAX Measures *(bonus)*
```dax
// Average ROI excluding anomaly weeks
Clean_Avg_ROI =
    CALCULATE(AVERAGE(Campaign_Performance[ROI_Pct]),
              Anomaly_Results[Risk_Level] = "NORMAL")

// Brand Risk Traffic Light
BRI_Status =
    SWITCH(TRUE(),
        [BRI_Score] < 30, "🟢 LOW",
        [BRI_Score] < 60, "🟡 MODERATE",
        "🔴 HIGH RISK")

// Rolling 4-week ROI
ROI_4W =
    AVERAGEX(
        DATESINPERIOD(Campaign_Performance[Date], LASTDATE(Campaign_Performance[Date]), -28, DAY),
        Campaign_Performance[ROI_Pct])
```

---

### Deliverables
- [ ] `.pbix` file with all 4 pages completed
- [ ] At least 1 custom DAX measure per page
- [ ] Cross-page drill-through working (click anomaly → see brand risk for that period)
- [ ] Published to Power BI Service (free account) with shareable link
- [ ] 2-minute screen recording walkthrough as CMO presenting to the board

---
##  Lab Summary & Reflection Questions

Run the cell below to print your final risk summary, then answer the reflection questions.


In [ ]:
print('=' * 65)
print('  MARKETING RISK & RESILIENCE LAB — FINAL SUMMARY')
print('=' * 65)
print(f'\n  Dataset:          {len(df)} weeks of advertising performance data')
print(f'  Mean ROI:         {df["ROI"].mean():.1f}%')
print(f'  ROI Std Dev:      {df["ROI"].std():.1f}%')
print()
print(f'  Monte Carlo VaR(5%):    {metrics["var_5"]:.1f}%  ← worst ROI in 5% of scenarios')
print(f'  Monte Carlo CVaR:       {metrics["cvar"]:.1f}%  ← avg loss in worst 5%')
print(f'  Probability of Loss:    {metrics["prob_loss"]:.1f}%')
print()
print(f'  Anomalies Detected:     {df_feat["is_anomaly"].sum()} weeks flagged by Isolation Forest')
print(f'  Brand Risk Index:       {bri_score:.1f}/100')
print()
status = '🟢 LOW RISK' if bri_score < 30 else ('🟡 MODERATE' if bri_score < 60 else '🔴 HIGH RISK')
print(f'  Overall Brand Status:   {status}')
print()
print('  Power BI Export:        marketing_risk_powerbi.xlsx ✅')
print('=' * 65)
print()
print('📝 Reflection Questions:')
print('  1. Which distribution did you choose for spend and why?')
print('  2. How would correlated risk factors change your VaR estimate?')
print('  3. What false positives did the anomaly model produce? Why?')
print('  4. How reliable is LLM risk scoring vs rule-based? Where does it fail?')
print('  5. What additional data would improve the Brand Risk Index?')

1. Which distribution did you choose for spend and why?

> Answer: I chose the Log-normal distribution (or Gamma distribution) for marketing spend.
Why: Marketing spend is never negative and is usually right-skewed. Most days or campaigns have moderate budgets, but there are occasional large spikes for major launches or seasonal events (like Black Friday). A Normal distribution wouldn't work here because it assumes spend could be negative and doesn't account for those "long tail" high-budget events.


2. How would correlated risk factors change your VaR estimate?
> Answer: Correlation significantly increases the VaR (Value at Risk) estimate.
Why: If risk factors are independent, one bad event might be offset by a good one. However, if they are correlated (e.g., a brand scandal happens at the same time as a market crash), the losses compound. In a correlated environment, the "worst-case scenario" becomes much more severe, meaning the potential loss (VaR) at the 5% threshold would be much higher than if the risks were diversified.


3. What false positives did the anomaly model produce? Why?
> Answer: The model produced false positives during planned high-activity periods, such as major holiday sales or viral product launches.
Why: Anomaly detection models look for statistical deviations from the trend. A massive spike in sales or spend due to a planned "Flash Sale" looks like an "error" to the model because it's so different from the daily average. Without external data (like a marketing calendar) being fed into the model, it cannot distinguish between a "bad" anomaly (a bug/fraud) and a "good" anomaly (a successful surprise sale).

4. How reliable is LLM risk scoring vs rule-based? Where does it fail?
> Answer: LLM risk scoring is superior for nuance, sarcasm, and context, while rule-based scoring is better for consistency and speed.
Reliability: LLMs are great at understanding why a post is risky (e.g., "This company is great at stealing money" — a rule-based system might see "great" and "money" as positive, while an LLM sees the irony).
Failure points: LLMs fail on "hallucinations" (making up facts) and token limits. They also lack a rigid "logical check"—an LLM might score the same text differently if you change one word, whereas a rule-based system is 100% predictable.

5. What additional data would improve the Brand Risk Index?
> Answer: To make the BRI more robust, I would add:
Velocity Data: How fast is a negative post being shared? (A post with 1,000 shares in 10 minutes is riskier than 1,000 shares in a week).
Competitor Benchmarking: Is the negative sentiment hitting just us, or the whole industry?
Google Trends/Search Volume: Are people actively searching for the "scandal"?
Customer Lifetime Value (CLV): Are our most loyal customers leaving, or just random "trolls"?
